# Archive: Comment Attaching — Script Version
<!-- structured-notebook -->
## Notebook Summary
Purpose: preserve the earlier scripted version of the comment-to-post linker. The canonical version is `05_comment_analysis/01_comment_attaching.ipynb`.

The script and notebook do the same thing (join Reddit comments to their parent posts and compute post-comment sentiment pairs). Keep this copy for batch-run reproducibility; use the notebook for interactive work.


In [ ]:
import pandas as pd
import numpy as np
import re

# --- Update these paths ---
SUBMISSIONS_PATH = "submissions.csv"   # or .jsonl
COMMENTS_PATH   = "comments.csv"       # your file

def read_any(path):
    if path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    return pd.read_csv(path)

subs = read_any(SUBMISSIONS_PATH)
coms = read_any(COMMENTS_PATH)

# Timestamps
for df in (subs, coms):
    if "created_utc" in df.columns:
        df["created_utc"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")

# Helpers
def strip_prefix(s):
    # remove t1_/t3_/etc prefixes from IDs
    return s.astype(str).str.replace(r"^t[0-6]_", "", regex=True)

# Post IDs in submissions
if "id" in subs.columns:
    subs["post_id"] = subs["id"].astype(str)

# Post IDs in comments (via link_id or permalink)
if "link_id" in coms.columns:
    coms["root_post_id"] = strip_prefix(coms["link_id"])
elif "permalink" in coms.columns:
    # /r/<sub>/comments/<POST_ID>/<slug>/<COMMENT_ID>/
    coms["root_post_id"] = coms["permalink"].str.extract(r"/comments/([a-z0-9]+)/", expand=False)
else:
    raise ValueError("No way to recover post id from comments: need link_id or permalink.")